In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [12]:
from langchain_groq import ChatGroq
model = ChatGroq(model ="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x141bacaf0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x141bac340>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [13]:
from langchain_core.messages import HumanMessage
model.invoke(
    [HumanMessage(content="Hello, My name is Rishabh I am a collge Student")]
)

AIMessage(content="Nice to meet you, Rishabh. It's great to hear that you're a college student. What's your major or field of study? Are you enjoying college life so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 49, 'total_tokens': 88, 'completion_time': 0.103158867, 'completion_tokens_details': None, 'prompt_time': 0.002610472, 'prompt_tokens_details': None, 'queue_time': 0.094723708, 'total_time': 0.105769339}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4e37-8cd9-7e92-bd0a-4935a1fa3d31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 39, 'total_tokens': 88})

In [16]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hello, My name is Rishabh I am a collge Student"),
        AIMessage(content="Nice to meet you, Rishabh. It's great to hear that you're a college student. What's your major or field of study? Are you enjoying college life so far?"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]   
)
# Since I am passing in a list of messages it can remember it

AIMessage(content="Your name is Rishabh, and you're a college student.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 108, 'total_tokens': 123, 'completion_time': 0.015997199, 'completion_tokens_details': None, 'prompt_time': 0.007026968, 'prompt_tokens_details': None, 'queue_time': 0.045163341, 'total_time': 0.023024167}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4e39-e302-73c2-8c7c-7122a601b1dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 15, 'total_tokens': 123})

 But there can be different sessions also like in chatgpt so it should remember wrt chat history

 We can use a message history class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them into datastore. Future interactions will then load those messages and pass them into chain as part of input


In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
# every message we give should be added in messagae history

# When different users are interacting with LLM how will we differentiate them
# for that we create a function

store={}

def get_session_history(session_id : str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]= ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)
# as soon as we invoke this it will take the model, the entire chat history using session_id
# and we will get entire history/ context

In [18]:
config = {"configurable":{"session_id": "chat1"}}

In [21]:
# I will use this session_id to chat with llm model
response = with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Rishabh I am a collge Student")],
    config=config
    # it means for this session_id I am interacting
)

In [22]:
response.content

"It seems like you accidentally repeated your introduction. Don't worry, let's start fresh. How can I assist you with anything today, Rishabh?"

In [ ]:
# let's say If i change the config will it be able to identify?
# changing the session_id
 
config1 = {"configurable":{"session_id":"chat1"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
    # Do you think it will be able to remember?
    # The reason it will remember is because I have given same session id = chat1
)
response.content

'Your name is Rishabh.'

In [26]:
# If I change the session id, it will not remember
config1 = {"configurable":{"session_id":"chat2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
    # Do you think it will be able to remember?
    # The reason it will remember is because I have given same session id = chat1
)
response.content

"I don't know your name. I'm a large language model, I don't have any information about you or your identity. This is the start of our conversation, and I don't retain any information from previous conversations. If you'd like to share your name, I'd be happy to chat with you!"

so now with with_message_history I will just give session id and it will remember